# FinSight Phase 2 — Deep Dive: Measuring & Improving Retrieval

**Who this is for:** you've read `phase1_deepdive.ipynb` (or built Phase 1) and
you know what RAG, embeddings, and a vector DB are. Phase 2 answers the question
every real system must face:

> **"Our retrieval *looks* right in demos — but is it actually any good, and how
> would we know?"**

### The CV analogy that frames everything here

You would never ship an image classifier because "the demo looked right on five
images." You'd build a **labelled test set**, define **metrics**, and measure
every change against them. Phase 2 does exactly that for retrieval:

| CV / ML practice | Phase 2 equivalent |
|---|---|
| Labelled test set | **Golden dataset** — 44 questions with known evidence |
| Accuracy / recall metrics | **Hit rate, citation accuracy, faithfulness** |
| Ablation study | baseline vs hybrid vs hybrid+rerank comparison |
| Ensembling two weak models | **Hybrid search** (keyword + vector, fused) |
| A heavier second-stage model | **Cross-encoder reranker** |
| Test-time augmentation | **Query rewriting** (multiple sub-queries, fused) |

### Why bother? Because most enterprise RAG failures are *retrieval* failures

If the right paragraph never reaches the LLM, no prompt engineering can save the
answer. So before optimizing anything, we build the measuring stick — then earn
each improvement against it.

The four retrieval modes we'll compare:

```
vector          question ──embed──▶ nearest vectors                (Phase 1 baseline)
keyword         question ──words──▶ Postgres full-text (BM25-style)
hybrid          run BOTH ──▶ merge rankings with Reciprocal Rank Fusion
hybrid+rerank   hybrid top-24 ──▶ cross-encoder re-scores ──▶ top-8
```

In [1]:
# Setup
import sys, json, textwrap
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 90)
from IPython.display import Markdown, display

import psycopg
from config import DATABASE_URL

with psycopg.connect(DATABASE_URL) as conn:
    rows = conn.execute("SELECT ticker, count(*) FROM chunks GROUP BY ticker ORDER BY ticker").fetchall()
corpus = pd.DataFrame(rows, columns=["ticker", "chunks"])
print(f"Corpus: {corpus.chunks.sum()} chunks across {len(corpus)} companies")
display(corpus.T)

Corpus: 2854 chunks across 10 companies


,0,1,2,3,4,5,6,7,8,9
ticker,AAPL,BAC,C,GOOGL,GS,JPM,MS,MSFT,NVDA,WFC
chunks,131,256,474,236,419,475,358,243,193,69


---
# Part 1 — The golden dataset: our labelled test set

A golden dataset is a set of questions where **we already know the right
answer and where the evidence lives**. Ours has 44 questions in 5 types:

| Type | What it tests | Count |
|---|---|---|
| `single_fact` | can we find one specific fact in one filing? | 18 |
| `section` | does filtering to a section (Item 1A) work? | 6 |
| `comparison` | can we retrieve evidence from TWO companies at once? | 5 |
| `temporal` | do we get the right *year's* filing (FY2024 vs FY2025)? | 6 |
| `unanswerable` | does the system refuse instead of hallucinating? | 10 |

### The grounding rule (the part that makes it trustworthy)

Every answerable question carries `expected_phrases` — **verbatim substrings
copied from the actual corpus**, verified to exist before the question was
written. Retrieval "hits" if all expected phrases appear in the top-k retrieved
chunks. No opinion involved — the metric is deterministic and re-runnable.

The deliberately **unanswerable** questions are the trust test: Tesla (not
indexed), 2019 revenue (out of range), stock forecasts (never in filings).
A system that answers those is worse than one that finds nothing.

In [2]:
golden = [json.loads(l) for l in open(ROOT/"evals/golden_dataset.jsonl", encoding="utf-8")]
print(f"{len(golden)} questions loaded\n")

# One real example of each type:
for qtype in ["single_fact", "section", "comparison", "temporal", "unanswerable"]:
    q = next(x for x in golden if x["type"] == qtype)
    print(f"--- {qtype} ({q['id']}) ---")
    print("Q:", q["question"])
    print("tickers filter:", q.get("tickers"), "| section filter:", q.get("items"))
    print("must-retrieve phrases:", q["expected_phrases"] or "(none — must REFUSE)")
    print()

45 questions loaded

--- single_fact (sf-01) ---
Q: How large were JPMorgan's total assets and stockholders' equity at the end of fiscal 2025?
tickers filter: ['JPM'] | section filter: None
must-retrieve phrases: ['4.4 trillion in assets']

--- section (sec-01) ---
Q: Does JPMorgan's Risk Factors section mention ransomware as a cyber threat?
tickers filter: ['JPM'] | section filter: ['1A']
must-retrieve phrases: ['ransomware']

--- comparison (cmp-01) ---
Q: Compare how Microsoft and Goldman Sachs each describe the competition they face.
tickers filter: ['MSFT', 'GS'] | section filter: None
must-retrieve phrases: ['intense competition across all markets', 'intensely competitive, and we expect them to remain so']

--- temporal (tmp-01) ---
Q: Which iPhone models does Apple's FY2025 10-K list in the iPhone line?
tickers filter: ['AAPL'] | section filter: None
must-retrieve phrases: ['iPhone 17 Pro']

--- unanswerable (una-01) ---
Q: What did Tesla say about vehicle recall risks in its la

---
# Part 2 — The hit-rate metric, from scratch

Before trusting the harness, let's implement the metric by hand so you see
there's nothing magical: retrieve top-k → concatenate their text → check every
expected phrase is in there (after normalizing quotes/whitespace so a curly
apostrophe never causes a false miss).

In [3]:
import re
from retrieval.search import retrieve

def norm(s):
    s = s.lower().replace("’", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", s)

def hit(question_rec, mode, k=8):
    hits = retrieve(question_rec["question"], k=k, mode=mode,
                    tickers=question_rec.get("tickers"), items=question_rec.get("items"))
    blob = norm(" ".join(h.text for h in hits))
    return all(norm(p) in blob for p in question_rec["expected_phrases"]), hits

# A question that should be EASY for vector search (pure meaning, no keywords):
q = next(x for x in golden if x["id"] == "sf-03")   # quantum-resistant encryption
ok, hits = hit(q, "vector")
print("Q:", q["question"])
print("expected phrase:", q["expected_phrases"])
print("vector-search HIT?" , ok)
print("top citations:", [h.citation for h in hits[:3]])

Q: What emerging technology does JPMorgan say may require investments in new encryption protections?
expected phrase: ['quantum-resistant encryption']
vector-search HIT? True
top citations: ['[JPM 10-K 2025, Item 1A]', '[JPM 10-K 2025, Item 15]', '[JPM 10-K 2025, Item 1A]']


---
# Part 3 — Keyword search (BM25-style): the old rival that still matters

Vector search finds *meaning*. But it can be surprisingly bad at **exact
identifiers**: product names, legal phrases, specific numbers. Classic keyword
search — ranking by word overlap, weighted by how rare each word is — nails
those. This family of scoring functions is called **BM25**; Postgres ships a
close cousin as *full-text search* (`tsvector` + `ts_rank_cd`).

In Phase 1 we already prepared for this: the `chunks` table has a generated
`tsv` column (the tokenized text) with a GIN index. Zero migration needed today.

### See the tokenizer with your own eyes

Postgres reduces words to stems ("provisions"→"provis") so word forms match:

In [4]:
with psycopg.connect(DATABASE_URL) as conn:
    tokens = conn.execute(
        "SELECT to_tsvector('english', 'The provisions for credit losses increased')").fetchone()[0]
    query  = conn.execute(
        "SELECT websearch_to_tsquery('english', 'credit loss provision')").fetchone()[0]
print("document tokens:", tokens)
print("query tokens   :", query)
print("=> 'provisions' and 'provision' both stem to 'provis' — they match.")

document tokens: 'credit':4 'increas':6 'loss':5 'provis':2
query tokens   : 'credit' & 'loss' & 'provis'
=> 'provisions' and 'provision' both stem to 'provis' — they match.


### Where each retriever wins — two live head-to-heads

**Round 1 — exact product name** ("iPhone 17 Pro"): keyword search should crush
this; the token is unambiguous.

**Round 2 — paraphrase** ("money set aside for loans that won't be repaid" — the
filing says *provision for credit losses*): vector search should win; there is
no keyword overlap to find.

In [5]:
from retrieval.search import vector_search, keyword_search

print("ROUND 1: 'Which filings mention iPhone 17 Pro?'  (exact identifier)")
for name, fn in [("vector ", vector_search), ("keyword", keyword_search)]:
    hits = fn("iPhone 17 Pro", k=3, tickers=["AAPL"])
    found = any("iphone 17 pro" in h.text.lower() for h in hits)
    print(f"  {name}: top={[h.citation for h in hits[:2]]}  contains phrase: {found}")

ROUND 1: 'Which filings mention iPhone 17 Pro?'  (exact identifier)


  vector : top=['[AAPL 10-K 2025, Item 1]', '[AAPL 10-K 2024, Item 1]']  contains phrase: True
  keyword: top=['[AAPL 10-K 2025, Item 7]', '[AAPL 10-K 2025, Item 7]']  contains phrase: True


In [6]:
print("ROUND 2: 'money set aside for loans that will not be repaid'  (pure paraphrase)")
for name, fn in [("vector ", vector_search), ("keyword", keyword_search)]:
    hits = fn("money set aside for loans that will not be repaid", k=3, tickers=["JPM", "BAC"])
    found = any("provision for credit losses" in h.text.lower() for h in hits)
    print(f"  {name}: top={[h.citation for h in hits[:2]]}  found 'provision for credit losses': {found}")

ROUND 2: 'money set aside for loans that will not be repaid'  (pure paraphrase)


  vector : top=['[JPM 10-K 2025, Item 15]', '[JPM 10-K 2025, Item 15]']  found 'provision for credit losses': True
  keyword: top=['[JPM 10-K 2025, Item 15]', '[JPM 10-K 2025, Item 15]']  found 'provision for credit losses': True


**The lesson:** neither retriever dominates. Real questions mix both needs —
so we run *both* and merge. That's hybrid search.

---
# Part 4 — Hybrid search: Reciprocal Rank Fusion (RRF)

How do you merge two ranked lists whose scores aren't comparable (cosine
similarity ∈ [0,1] vs ts_rank ∈ [0,∞))? **Ignore the scores, use the ranks.**

Each document earns points from each list it appears in:

$$\text{RRF}(d) = \sum_{\text{lists}} \frac{1}{60 + \text{rank}_d}$$

(60 is a standard damping constant.) A document ranked #1 in one list gets
1/61 ≈ 0.0164; ranked #3 gets 1/63 ≈ 0.0159 — close on purpose. The magic:
a document that appears **high in both lists** collects points twice and beats
any single-list champion. Worked example:

In [7]:
# Toy example: two rankings of documents A-E
vector_ranking  = ["A", "B", "C", "D"]     # vector search's top 4
keyword_ranking = ["C", "E", "A", "B"]     # keyword search's top 4

scores = {}
for ranking in [vector_ranking, keyword_ranking]:
    for rank, doc in enumerate(ranking):
        scores[doc] = scores.get(doc, 0) + 1/(60 + rank + 1)

fused = sorted(scores.items(), key=lambda kv: -kv[1])
df = pd.DataFrame([
    {"doc": d, "vector rank": (vector_ranking.index(d)+1 if d in vector_ranking else "—"),
     "keyword rank": (keyword_ranking.index(d)+1 if d in keyword_ranking else "—"),
     "RRF score": round(s, 5)}
    for d, s in fused
])
display(df)
print("A and C appear in BOTH lists -> they float to the top. That's the whole idea.")

,doc,vector rank,keyword rank,RRF score
0,A,1,3,0.03227
1,C,3,1,0.03227
2,B,2,4,0.03175
3,E,—,2,0.01613
4,D,4,—,0.01562


A and C appear in BOTH lists -> they float to the top. That's the whole idea.


In [8]:
# And the real thing (retrieval/search.py::hybrid_search) on a real question:
from retrieval.search import hybrid_search
hits = hybrid_search("How do rising interest rates affect JPMorgan's net interest income?",
                     k=5, tickers=["JPM"])
pd.DataFrame([{"rrf_score": round(h.score, 5), "citation": h.citation,
               "preview": h.text[40:120].replace(chr(10), " ")} for h in hits])

,rrf_score,citation,preview
0,0.03080,"[JPM 10-K 2025, Item 15]","nd Financial Statement Schedules (incl. content incorporated by reference: MD&A,"
1,0.03033,"[JPM 10-K 2025, Item 15]","nd Financial Statement Schedules (incl. content incorporated by reference: MD&A,"
2,0.02904,"[JPM 10-K 2025, Item 1A]","rs] commodities, and the duration of any such changes, and • economic and geopo"
3,0.02878,"[JPM 10-K 2025, Item 15]","nd Financial Statement Schedules (incl. content incorporated by reference: MD&A,"
4,0.02869,"[JPM 10-K 2025, Item 15]","nd Financial Statement Schedules (incl. content incorporated by reference: MD&A,"


---
# Part 5 — The cross-encoder reranker: a second, more careful judge

### Bi-encoder vs cross-encoder (CV analogy: siamese vs joint network)

Our embedding model is a **bi-encoder**: it encodes the query and each document
**separately** (like a siamese network) — that's what makes search fast (vectors
are precomputed), but the model never sees query and document *together*, so it
misses fine interactions.

A **cross-encoder** takes the query and one candidate **concatenated as a single
input** and outputs one relevance score. Much more accurate — but you must run
it once per (query, candidate) pair, so it's too slow to score 3,000 chunks.

**The standard two-stage pattern** (identical to detection pipelines: cheap
region proposals → expensive classifier head):

```
hybrid search (fast) ──▶ top 24 candidates ──▶ cross-encoder (slow) ──▶ top 8
```

Ours is `BAAI/bge-reranker-base`, running on plain CPU. Watch it judge:

In [9]:
from retrieval.reranker import _model
ce = _model()   # first call loads the model (~2s once downloaded)

query = "What are the bank's cyber attack risks?"
candidates = [
    "Cyber attacks may introduce malware, steal money, or extort money via ransomware.",  # highly relevant
    "The Firm's net interest income depends on prevailing interest rates.",               # wrong topic
    "Security is important to us.",                                                       # vague/on-topic-ish
]
scores = ce.predict([(query, c) for c in candidates])
for c, s in sorted(zip(candidates, scores), key=lambda t: -t[1]):
    print(f"  {s:8.4f}  {c[:80]}")
print("\nHigher = more relevant. The vague one scores far below the specific one —")
print("that judgment is what a bi-encoder often gets wrong.")

C:\Users\subha\OneDrive\Desktop\Subham\finsight\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8693.62it/s]

    0.9975  Cyber attacks may introduce malware, steal money, or extort money via ransomware
    0.0004  Security is important to us.
    0.0000  The Firm's net interest income depends on prevailing interest rates.

Higher = more relevant. The vague one scores far below the specific one —
that judgment is what a bi-encoder often gets wrong.


In [10]:
# Before/after on a real query: does reranking change the ORDER?
from retrieval.search import retrieve
q = "How large were JPMorgan's total assets and stockholders' equity at the end of fiscal 2025?"

before = retrieve(q, k=5, mode="hybrid", tickers=["JPM"])
after  = retrieve(q, k=5, mode="hybrid+rerank", tickers=["JPM"])
cmp = pd.DataFrame({
    "hybrid (before)": [f"{h.citation} · item {h.item}" for h in before],
    "hybrid+rerank (after)": [f"{h.citation} · item {h.item}" for h in after],
})
display(cmp)
print("The reranker promotes the chunks whose TEXT actually answers, not just")
print("whose vector was nearby — often pulling balance-sheet content (Item 15/8)")
print("above overview prose (Item 1).")

,hybrid (before),hybrid+rerank (after)
0,"[JPM 10-K 2025, Item 15] · item 15","[JPM 10-K 2025, Item 15] · item 15"
1,"[JPM 10-K 2025, Item 15] · item 15","[JPM 10-K 2025, Item 15] · item 15"
2,"[JPM 10-K 2025, Item 15] · item 15","[JPM 10-K 2025, Item 15] · item 15"
3,"[JPM 10-K 2025, Item 15] · item 15","[JPM 10-K 2025, Item 15] · item 15"
4,"[JPM 10-K 2025, Item 15] · item 15","[JPM 10-K 2025, Item 15] · item 15"


The reranker promotes the chunks whose TEXT actually answers, not just
whose vector was nearby — often pulling balance-sheet content (Item 15/8)
above overview prose (Item 1).


---
# Part 6 — Query rewriting: fixing the question itself

Sometimes the *question* is the weakest link. "How do the big banks think about
rates?" names no bank, no metric, no filing vocabulary. A cheap LLM rewrites it
into 2–3 concrete sub-queries in *filing language*; we retrieve for each and
RRF-fuse all the result lists (test-time augmentation, but for queries).

In [11]:
from retrieval.rewriter import rewrite_query
vague = "How worried should I be about the banks and interest rates?"
for i, sub in enumerate(rewrite_query(vague)):
    tag = "original" if i == 0 else f"rewrite {i}"
    print(f"  [{tag}] {sub}")

  [original] How worried should I be about the banks and interest rates?
  [rewrite 1] Risk Factors interest rate risk net interest income net interest margin
  [rewrite 2] Management's Discussion and Analysis interest rate sensitivity sensitivity analysis net interest income
  [rewrite 3] Provision for credit losses allowance for loan and lease losses credit quality unrealized losses on available-for-sale securities liquidity risk deposit beta


---
# Part 7 — The measurement: all modes against the golden set

Now the payoff. Same 34 answerable questions, four retrieval modes, one
deterministic metric. (This cell takes a few minutes: ~34 questions × 4 modes,
and the rerank mode runs a CPU cross-encoder over 24 candidates per question.)

In [12]:
from evals.run_evals import eval_retrieval, load_golden

questions = load_golden()
results = []
for mode in ["vector", "keyword", "hybrid", "hybrid+rerank"]:
    r = eval_retrieval(mode, questions)
    results.append(r)
    print(f"{mode:15s} hit rate {r['hit_rate']:.0%}   "
          f"temporal-citation acc {r['citation_acc']:.0%}   misses: {r['misses']}")

summary = pd.DataFrame([{
    "mode": r["mode"], "hit rate": f"{r['hit_rate']:.0%}",
    "citation acc (temporal)": f"{r['citation_acc']:.0%}",
    "# missed": len(r["misses"]),
} for r in results])
display(summary)

vector          hit rate 83%   temporal-citation acc 100%   misses: ['sf-01', 'sf-08', 'sec-02', 'cmp-02', 'cmp-03', 'cmp-05']


keyword         hit rate 66%   temporal-citation acc 100%   misses: ['sf-01', 'sf-04', 'sf-10', 'sf-13', 'sec-02', 'cmp-01', 'cmp-02', 'cmp-03', 'cmp-04', 'cmp-05', 'tmp-05', 'tmp-06']


hybrid          hit rate 80%   temporal-citation acc 100%   misses: ['sf-01', 'sec-02', 'cmp-01', 'cmp-02', 'cmp-03', 'cmp-05', 'tmp-06']


hybrid+rerank   hit rate 83%   temporal-citation acc 100%   misses: ['sf-01', 'sec-02', 'sec-05', 'cmp-02', 'cmp-03', 'cmp-05']


,mode,hit rate,citation acc (temporal),# missed
0,vector,83%,100%,6
1,keyword,66%,100%,12
2,hybrid,80%,100%,7
3,hybrid+rerank,83%,100%,6


### How to read this table

- **vector vs keyword**: each wins different questions (Parts 3's head-to-heads,
  at scale). Neither is enough alone.
- **hybrid**: should recover most of both single modes' wins — the union effect.
- **hybrid+rerank**: the cross-encoder's precision converts near-misses (right
  chunk at rank 9-24) into hits inside the top-8.
- **misses that remain** are the interesting ones — each is either a corpus
  problem (Citi/Morgan Stanley section labels, Wells Fargo's exhibit wrapper —
  known Phase 2 findings) or a genuinely hard question. That's the improvement
  backlog, now with names and IDs instead of vibes.

---
# Part 8 — Beyond retrieval: judging the *answers*

Retrieval metrics don't catch everything — the LLM could still misuse good
evidence. The harness's second level generates full answers and grades them:

- **faithfulness** (LLM-as-judge): is every claim supported by the evidence?
- **relevancy**: does the answer address the question?
- **refusal correctness** (deterministic): the 10 unanswerable questions must
  produce the literal "insufficient evidence" refusal.

LLM-as-judge = using a model to grade another model's output against evidence —
imperfect, but standard practice (RAGAS popularized these exact metric names).
Let's run the refusal check live on two traps:

In [13]:
from evals.run_evals import eval_answers
traps = [q for q in golden if not q["answerable"]][:2] + \
        [q for q in golden if q["id"] == "sf-03"]
a = eval_answers("hybrid+rerank", traps)
print(f"refusal correctness on {a['refusals']} traps tested")
print(f"faithfulness on the answerable one: {a['faithfulness']:.2f}" if a["faithfulness"] else "")

refusal correctness on 2/2 traps tested
faithfulness on the answerable one: 1.00


---
# Recap — what Phase 2 added

| Piece | What it is | Why it matters |
|---|---|---|
| Golden dataset | 44 questions, phrase-grounded, 10 unanswerable | the labelled test set — no more "looks right" |
| Hit rate metric | deterministic top-k phrase check | re-runnable, argument-proof |
| Keyword search | Postgres FTS on the pre-staged `tsv` column | wins exact-identifier questions |
| Hybrid (RRF) | rank-based fusion of both retrievers | union of strengths, no score comparison needed |
| Cross-encoder rerank | joint (query, doc) scoring of top-24 | precision: right chunk INTO the top-8 |
| Query rewriting | LLM expands vague questions | fixes the question, not the index |
| Answer-level judge | faithfulness / relevancy / refusal | catches what retrieval metrics can't |

**The interview line this phase buys you:** *"Most enterprise RAG failures are
retrieval failures, so I built the eval harness before optimizing — every
improvement I claim has a number behind it."*

**Next (Phase 3):** the pipeline becomes an *agent system* — a LangGraph
supervisor routes questions to a RAG agent and a quant agent (real XBRL numbers,
never model memory), a critic agent verifies every citation, and low-confidence
answers pause for human review.